In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from src.eval.llama_evaluator import LLAMA2_PROMPT

texts = [
    ("hi!", "Hello! How can I assist you today?"),
    ("What is the capital of France?", "The capital of France is Paris."),
    ("Can you tell me a joke?", "Sure! Why don't scientists trust atoms? Because they make up everything!"),
    ("What is the meaning of life?", "The meaning of life is subjective and can vary from person to person."),
    ("Tell me a story.", "Once upon a time in a faraway land, there lived a brave knight who fought dragons and saved villages."),
    (
        "How do I cook pasta?",
        "To cook pasta, boil water, add salt, then add the pasta and cook until al dente. Drain and serve with your favorite sauce.",
    ),
]

formated = [LLAMA2_PROMPT.format(behavior=prm, generation=gen) for (prm, gen) in texts]

conversations = [[{"role": "user", "content": LLAMA2_PROMPT.format(behavior=prm, generation=gen)}] for (prm, gen) in texts]

In [ ]:
from gserve import LLMConfig, ServeConfig
from gserve.vllm_service import VLLMService
from vllm import SamplingParams

model_name = "meta-llama/Llama-3.1-8B-Instruct"

llm_config = LLMConfig(
    model_name=model_name,
    dtype="bfloat16",
)

serve_config = ServeConfig(gpu_ids=[1], startup_timeout=10 * 60, client_timeout=2 * 60, verbose=True)

model = VLLMService(llm_config, serve_config)
model.start()

In [ ]:
sampling_params = SamplingParams(
    temperature=0,
    top_p=1.0,
    top_k=-1,
    repetition_penalty=1.0,
    max_tokens=40,
)

output = model.generate(formated, sampling_params=sampling_params)

for out in output:
    print(out)

In [ ]:
# now lets use transformers to do the same
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers

tokenizer = AutoTokenizer.from_pretrained(model_name)
model_transformers = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

In [ ]:
pipeline = transformers.pipeline(
    "text-generation",
    model=model_transformers,
    tokenizer=tokenizer,
    model_kwargs={"torch_dtype": torch.bfloat16, "skip_special_tokens": True, "add_generation_prompt": True, "padding_side": "left"},
    max_new_tokens=40,
    device="cuda:0",
    pad_token_id=128001,
    eos_token_id=128001,
)

batch_output = pipeline(
    formated,
    return_full_text=False,
    do_sample=False,
    top_p=1.0,
    top_k=-1,
    temperature=0,
    use_model_defaults=True,
    pad_token_id=128001,
    eos_token_id=128001,
    repetition_penalty=1,
)

for out in batch_output:
    print(out)

In [ ]:
raise "aaaaa"

----------------------

In [ ]:
import torch
import numpy as np

# 1. Common settings
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
SEED = 42
TEMPERATURE = 0.0
TOP_K = 0
TOP_P = 1.0
REPETITION_PENALTY = 1.1
MAX_NEW_TOKENS = 128
PROMPT = "How do you do, where are you, blah afaf comlet, "

# fix seeds for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# 2. vLLM setup
from vllm import LLM, SamplingParams

vllm_llm = LLM(
    model=MODEL_NAME,
    seed=SEED,
    dtype="bfloat16",
)

In [ ]:
vllm_sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY,
    max_tokens=MAX_NEW_TOKENS,
)

# generate with vLLM
vllm_resp = vllm_llm.generate(
    PROMPT,
    sampling_params=vllm_sampling_params,
)

vllm_output = vllm_resp[0].outputs[0].text
print("vLLM output:\n", vllm_output)

In [ ]:
# 3. Transformers pipeline setup
from transformers import AutoTokenizer, AutoModelForCausalLM, TextGenerationPipeline, GenerationConfig

# load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    add_generation_prompt=True,
    padding_side="left",
    use_fast=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="cuda:1",
)

# ensure pad_token is set
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
# generation config matching vLLM SamplingParams
gen_config = GenerationConfig(
    do_sample=False,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY,
    max_new_tokens=MAX_NEW_TOKENS,
)

# create a torch Generator with the same seed
generator = torch.Generator(device=model.device).manual_seed(SEED)

pipeline = TextGenerationPipeline(
    model=model.eval(),
    tokenizer=tokenizer,
    generation_config=gen_config,
)

# generate with transformers
hf_outputs = pipeline(
    PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY,
    num_return_sequences=1,
    return_full_text=True,
    use_model_defaults=True,
)

hf_output = hf_outputs[0]["generated_text"]
print("\nTransformers output:\n", hf_output)